# 🌊 MarineGuard AI: SAR Oil Spill Segmentation & Training Pipeline
### Smart India Hackathon 2026 (Problem Statement PS-1655)

This notebook demonstrates the end-to-end Machine Learning pipeline for **MarineGuard AI**:
1. **NOAA NESDIS & Sentinel-1 SAR Dataset Ingestion & EDA**
2. **Adaptive Lee Speckle Filtering & GLCM Haralick Textural Feature Extraction**
3. **Attention U-Net Architecture Implementation in PyTorch**
4. **Combined Dice + BCE Loss Training Loop**
5. **Quantitative Benchmark Evaluation (IoU, Dice/F1, Precision, Recall, Confusion Matrix)**

In [ ]:
import os
import sys
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

# Add root directory to sys.path
sys.path.insert(0, os.path.abspath('..'))

from ml.models.unet import SARUNet, DiceBCELoss
from ml.dataset.noaa_loader import NOAAOilSpillDataset, generate_noaa_benchmark_dataset
from ml.inference import apply_lee_filter, extract_glcm_features
from ml.evaluate import evaluate_model

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 1. NOAA & Sentinel-1 SAR Benchmark Dataset Generation

In [ ]:
data_dir = "../data/sar"
metadata = generate_noaa_benchmark_dataset(data_dir, num_samples=40)
print(f"Successfully initialized {len(metadata)} NOAA SAR benchmark records.")

## 2. SAR Preprocessing: Adaptive Lee Filter & GLCM Texture

In [ ]:
# Load sample image
img_path = glob.glob(os.path.join(data_dir, "images", "*.png"))[0]
mask_path = img_path.replace("images", "masks").replace(".png", "_mask.png")

raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
lee_img = apply_lee_filter(raw_img, window_size=7)
mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

glcm_metrics = extract_glcm_features(raw_img, mask)
print("GLCM Haralick Textural Descriptors:", glcm_metrics)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_img, cmap='gray')
axes[0].set_title("Raw SAR Backscatter")
axes[1].imshow(lee_img, cmap='gray')
axes[1].set_title("Lee Speckle Filtered")
axes[2].imshow(mask, cmap='hot')
axes[2].set_title("Ground Truth Spill Mask")
plt.tight_layout()
plt.show()

## 3. PyTorch Attention U-Net Model Verification

In [ ]:
model = SARUNet(n_channels=3, n_classes=1, bilinear=True, use_attention=True)
dummy_input = torch.randn(2, 3, 256, 256)
output = model(dummy_input)
print(f"Model forward pass successful! Input shape: {dummy_input.shape} -> Output shape: {output.shape}")

## 4. Benchmark Evaluation & Metrics

In [ ]:
report = evaluate_model(model_path="../models/unet_sar_oilspill.pth", data_dir="../data/sar", output_report="../ml/evaluation_report.json")
print("Benchmark evaluation completed.")